In [ ]:
# Questão 1

import random

# BUSCAS COM CONTAGEM

def linear_search(arr, target):
    comparacoes = 0
    for i in range(len(arr)):
        comparacoes += 1
        if arr[i] == target:
            return i, comparacoes
    return -1, comparacoes


def binary_search(arr, target):
    comparacoes = 0
    left, right = 0, len(arr) - 1

    while left <= right:
        mid = (left + right) // 2
        comparacoes += 1

        if arr[mid] == target:
            return mid, comparacoes
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1

    return -1, comparacoes



# GERADOR DE VETORES


def gerar_vetor(n):
    return [random.randint(0, n) for _ in range(n)]


# PRÉ-CONDIÇÃO (ARRAY ORDENADO)

def esta_ordenado(arr):
    # Falha rápida: verifica apenas algumas posições (amostragem)
    comparacoes = 0
    passo = max(1, len(arr) // 10)  # pega 10 pontos

    for i in range(0, len(arr) - passo, passo):
        comparacoes += 1
        if arr[i] > arr[i + passo]:
            return False, comparacoes

    return True, comparacoes


def binary_search_com_verificacao(arr, target):
    ordenado, comp_check = esta_ordenado(arr)

    if not ordenado:
        print("Vetor NÃO ordenado! Falha rápida ativada.")
        return -1, comp_check

    pos, comp_busca = binary_search(arr, target)
    return pos, comp_check + comp_busca


# TESTES EM ESCALAS


def testar():
    tamanhos = [100, 1000, 10000, 100000, 1000000]

    for n in tamanhos:
        print(f"\n===== Teste com n = {n} =====")

        vetor = gerar_vetor(n)
        alvo = vetor[n // 2]  # garante que existe

        # Linear Search
        pos_l, comp_l = linear_search(vetor, alvo)
        print(f"Linear Search -> Posição: {pos_l}, Comparações: {comp_l}")

        # Binary Search (com ordenação)
        vetor_ordenado = sorted(vetor)
        pos_b, comp_b = binary_search_com_verificacao(vetor_ordenado, alvo)
        print(f"Binary Search -> Posição: {pos_b}, Comparações: {comp_b}")



# EXECUÇÃO

if __name__ == "__main__":
    testar()


===== Teste com n = 100 =====
Linear Search -> Posição: 50, Comparações: 51
Binary Search -> Posição: 74, Comparações: 11

===== Teste com n = 1000 =====
Linear Search -> Posição: 500, Comparações: 501
Binary Search -> Posição: 305, Comparações: 18

===== Teste com n = 10000 =====
Linear Search -> Posição: 811, Comparações: 812
Binary Search -> Posição: 630, Comparações: 21

===== Teste com n = 100000 =====
Linear Search -> Posição: 8759, Comparações: 8760
Binary Search -> Posição: 71394, Comparações: 24

===== Teste com n = 1000000 =====
Linear Search -> Posição: 486054, Comparações: 486055
Binary Search -> Posição: 763009, Comparações: 27


Estratégia de “falha rápida”

Ao invés de verificar todo o vetor (O(n)), usamos amostragem:

Verifica alguns pontos espaçados do vetor (ex: 10 posições)
Se encontrar um erro → já falha
Se não encontrar → assume que provavelmente está ordenado
Justificativa (Big O)
Verificação completa: O(n)
Falha rápida (amostragem): O(k) → onde k é constante (ex: 10)

Portanto:

Muito mais rápido que ordenar (O(n log n))
Muito mais barato que validar tudo
Risco de falso positivo
Pode acontecer se:
O vetor estiver quase ordenado
O erro estiver em regiões não verificadas

Ou seja:

Existe risco, mas é pequeno na prática
É um trade-off entre velocidade vs precisão

In [ ]:
# Questão 2

import random
import sys
sys.setrecursionlimit(100000)


# GERADORES DE VETORES

def vetor_ordenado(n):
    return list(range(n))

def vetor_reverso(n):
    return list(range(n, 0, -1))

def vetor_quase_ordenado(n):
    arr = list(range(n))
    for _ in range(n // 10):  # bagunça 10%
        i = random.randint(0, n - 1)
        j = random.randint(0, n - 1)
        arr[i], arr[j] = arr[j], arr[i]
    return arr

def vetor_aleatorio(n):
    return [random.randint(0, n) for _ in range(n)]


# ALGORITMOS DE ORDENAÇÃO


def bubble_sort(arr):
    comp = 0
    copias = 0
    n = len(arr)

    for i in range(n):
        for j in range(0, n - i - 1):
            comp += 1
            if arr[j] > arr[j + 1]:
                # troca = 3 cópias
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                copias += 3

    return comp, copias


def selection_sort(arr):
    comp = 0
    copias = 0
    n = len(arr)

    for i in range(n):
        min_idx = i
        for j in range(i + 1, n):
            comp += 1
            if arr[j] < arr[min_idx]:
                min_idx = j

        if min_idx != i:
            arr[i], arr[min_idx] = arr[min_idx], arr[i]
            copias += 3

    return comp, copias


def insertion_sort(arr):
    comp = 0
    copias = 0

    for i in range(1, len(arr)):
        chave = arr[i]
        j = i - 1

        while j >= 0:
            comp += 1
            if arr[j] > chave:
                arr[j + 1] = arr[j]
                copias += 1
                j -= 1
            else:
                break

        arr[j + 1] = chave
        copias += 1

    return comp, copias


# BST SORT

class Node:
    def __init__(self, valor):
        self.valor = valor
        self.esq = None
        self.dir = None


class BST:
    def __init__(self):
        self.root = None
        self.comp = 0
        self.chamadas = 0

    def inserir(self, node, valor):
        if node is None:
            return Node(valor)

        self.comp += 1
        if valor < node.valor:
            node.esq = self.inserir(node.esq, valor)
        else:
            node.dir = self.inserir(node.dir, valor)

        return node

    def inserir_lista(self, arr):
        for v in arr:
            self.root = self.inserir(self.root, v)

    def in_order(self, node, resultado):
        if node:
            self.chamadas += 1
            self.in_order(node.esq, resultado)
            resultado.append(node.valor)
            self.in_order(node.dir, resultado)

    def ordenar(self):
        resultado = []
        self.in_order(self.root, resultado)
        return resultado



# TESTES

def testar():
    tamanhos = [1000, 5000] # Deixei pouco chamados para teste mas se quiser ver tudo só copiar e mandar ali [1000, 10000, 25000, 50000, 100000]
    tipos = {
        "Ordenado": vetor_ordenado,
        "Reverso": vetor_reverso,
        "Quase Ordenado": vetor_quase_ordenado,
        "Aleatório": vetor_aleatorio
    }

    for n in tamanhos:
        print(f"\n========== TAMANHO: {n} ==========")

        for nome, func in tipos.items():
            print(f"\n--- Vetor {nome} ---")

            vetor = func(n)

            # Bubble
            v1 = vetor.copy()
            comp, cop = bubble_sort(v1)
            print(f"Bubble -> Comp: {comp}, Copias: {cop}")

            # Selection
            v2 = vetor.copy()
            comp, cop = selection_sort(v2)
            print(f"Selection -> Comp: {comp}, Copias: {cop}")

            # Insertion
            v3 = vetor.copy()
            comp, cop = insertion_sort(v3)
            print(f"Insertion -> Comp: {comp}, Copias: {cop}")

            # BST
            bst = BST()
            bst.inserir_lista(vetor)
            bst.ordenar()
            print(f"BST -> Comp: {bst.comp}, Chamadas: {bst.chamadas}")


if __name__ == "__main__":
    testar()


========== TAMANHO: 1000 ==========

--- Vetor Ordenado ---
Bubble -> Comp: 499500, Copias: 0
Selection -> Comp: 499500, Copias: 0
Insertion -> Comp: 999, Copias: 999
BST -> Comp: 499500, Chamadas: 1000

--- Vetor Reverso ---
Bubble -> Comp: 499500, Copias: 1498500
Selection -> Comp: 499500, Copias: 1500
Insertion -> Comp: 499500, Copias: 500499
BST -> Comp: 499500, Chamadas: 1000

--- Vetor Quase Ordenado ---
Bubble -> Comp: 499500, Copias: 176112
Selection -> Comp: 499500, Copias: 300
Insertion -> Comp: 59703, Copias: 59703
BST -> Comp: 37867, Chamadas: 1000

--- Vetor Aleatório ---
Bubble -> Comp: 499500, Copias: 754767
Selection -> Comp: 499500, Copias: 2985
Insertion -> Comp: 252583, Copias: 252588
BST -> Comp: 10901, Chamadas: 1000

========== TAMANHO: 5000 ==========

--- Vetor Ordenado ---
Bubble -> Comp: 12497500, Copias: 0
Selection -> Comp: 12497500, Copias: 0
Insertion -> Comp: 4999, Copias: 4999
BST -> Comp: 12497500, Chamadas: 5000

--- Vetor Reverso ---
Bubble -> Comp: 

Comparação dos algoritmos


Bubble Sort
Sempre faz muitas comparações
Melhor caso: O(n²)
Pior caso: O(n²)
Muito lento


Selection Sort
Comparações fixas: O(n²)
Poucas trocas
Não melhora com vetor ordenado


Insertion Sort
Melhor caso: O(n) (vetor ordenado)
Pior caso: O(n²)
Excelente para vetores quase ordenados


BST Sort
Melhor/Médio: O(n log n)
Pior caso: O(n²) (quando vira lista — vetor ordenado)

In [ ]:
# Questão 3

import random
import time


# GERAR VETORES (com duplicados)

def gerar_vetor(n):
    return [random.randint(0, n // 2) for _ in range(n)]  # força duplicados


# REMOVER DUPLICADOS O(n)

def remover_duplicados(arr):
    vistos = set()
    resultado = []

    for x in arr:
        if x not in vistos:
            vistos.add(x)
            resultado.append(x)

    return resultado


# K MENORES - VERSÃO A (ORDENAÇÃO)

def k_smallest_sort(arr, k):
    arr_ordenado = sorted(arr)
    return arr_ordenado[:k]


# QUICKSELECT

def particionar(arr, low, high):
    pivot = arr[high]
    i = low

    for j in range(low, high):
        if arr[j] <= pivot:
            arr[i], arr[j] = arr[j], arr[i]
            i += 1

    arr[i], arr[high] = arr[high], arr[i]
    return i


def quickselect(arr, low, high, k):
    if low <= high:
        pi = particionar(arr, low, high)

        if pi == k:
            return
        elif pi < k:
            quickselect(arr, pi + 1, high, k)
        else:
            quickselect(arr, low, pi - 1, k)


def k_smallest_quickselect(arr, k):
    arr_copy = arr.copy()
    quickselect(arr_copy, 0, len(arr_copy) - 1, k)
    return arr_copy[:k]


# TESTES

def testar():
    tamanhos = [1000, 10000, 25000, 50000, 100000]
    k = 10

    for n in tamanhos:
        print(f"\n========== TAMANHO: {n} ==========")

        vetor = gerar_vetor(n)

        # Remover duplicados
        inicio = time.time()
        sem_dup = remover_duplicados(vetor)
        tempo_dup = time.time() - inicio

        print(f"Remoção duplicados -> Tamanho final: {len(sem_dup)} | Tempo: {tempo_dup:.5f}s")

        # Versão A (sort)
        inicio = time.time()
        menores_a = k_smallest_sort(sem_dup, k)
        tempo_a = time.time() - inicio

        print(f"Sort -> k menores: {menores_a[:5]}... | Tempo: {tempo_a:.5f}s")

        # Versão B (QuickSelect)
        inicio = time.time()
        menores_b = k_smallest_quickselect(sem_dup, k)
        tempo_b = time.time() - inicio

        print(f"QuickSelect -> k menores: {menores_b[:5]}... | Tempo: {tempo_b:.5f}s")


if __name__ == "__main__":
    testar()


========== TAMANHO: 1000 ==========
Remoção duplicados -> Tamanho final: 433 | Tempo: 0.00012s
Sort -> k menores: [0, 1, 2, 3, 5]... | Tempo: 0.00006s
QuickSelect -> k menores: [5, 1, 2, 3, 10]... | Tempo: 0.00019s

========== TAMANHO: 10000 ==========
Remoção duplicados -> Tamanho final: 4340 | Tempo: 0.00118s
Sort -> k menores: [0, 1, 2, 3, 4]... | Tempo: 0.00062s
QuickSelect -> k menores: [0, 1, 5, 6, 10]... | Tempo: 0.00125s

========== TAMANHO: 25000 ==========
Remoção duplicados -> Tamanho final: 10796 | Tempo: 0.00156s
Sort -> k menores: [0, 1, 2, 3, 4]... | Tempo: 0.00140s
QuickSelect -> k menores: [3, 2, 0, 1, 4]... | Tempo: 0.00078s

========== TAMANHO: 50000 ==========
Remoção duplicados -> Tamanho final: 21635 | Tempo: 0.00486s
Sort -> k menores: [0, 1, 2, 3, 4]... | Tempo: 0.00276s
QuickSelect -> k menores: [0, 1, 2, 3, 4]... | Tempo: 0.00174s

========== TAMANHO: 100000 ==========
Remoção duplicados -> Tamanho final: 43347 | Tempo: 0.01047s
Sort -> k menores: [0, 2, 3, 4

Explicação pronta 


Remoção de duplicados
Usa set para verificar elementos já vistos
Complexidade: O(n)
Muito eficiente


Versão A (ordenar tudo)
Ordena o vetor completo
Complexidade: O(n log n)
Simples, porém mais lento


Versão B (QuickSelect)
Não ordena tudo
Apenas encontra os k menores
Complexidade média: O(n)
Pior caso: O(n²)

In [ ]:
# Questão 4

class Node:
    def __init__(self, key, value):
        self.key = key
        self.value = value
        self.next = None


class HashTableChained:
    def __init__(self, capacidade=10):
        self.capacidade = capacidade
        self.tabela = [None] * capacidade
        self.tamanho = 0
        self.comparacoes = 0  # contador de comparações

    def _hash(self, key):
        return hash(key) % self.capacidade

    def __len__(self):
        return self.tamanho


    # PUT (INSERIR/ATUALIZAR)
 
    def put(self, key, value):
        indice = self._hash(key)
        atual = self.tabela[indice]

        # verifica se já existe (update)
        while atual:
            self.comparacoes += 1
            if atual.key == key:
                atual.value = value
                return
            atual = atual.next

        # insere novo nó
        novo = Node(key, value)
        novo.next = self.tabela[indice]
        self.tabela[indice] = novo
        self.tamanho += 1

        # verifica fator de carga
        if self.tamanho / self.capacidade > 0.75:
            self._rehash()


    # GET

    def get(self, key):
        indice = self._hash(key)
        atual = self.tabela[indice]

        while atual:
            self.comparacoes += 1
            if atual.key == key:
                return atual.value
            atual = atual.next

        return None


    # DELETE

    def delete(self, key):
        indice = self._hash(key)
        atual = self.tabela[indice]
        anterior = None

        while atual:
            self.comparacoes += 1
            if atual.key == key:
                if anterior:
                    anterior.next = atual.next
                else:
                    self.tabela[indice] = atual.next
                self.tamanho -= 1
                return True
            anterior = atual
            atual = atual.next

        return False

   
    # REHASH

    def _rehash(self):
        print("Rehash acontecendo...")

        antiga = self.tabela
        self.capacidade *= 2
        self.tabela = [None] * self.capacidade
        self.tamanho = 0

        for head in antiga:
            atual = head
            while atual:
                self.put(atual.key, atual.value)
                atual = atual.next



# TESTES


def testar():
    ht = HashTableChained()

    print("Inserindo elementos...")
    for i in range(20):
        ht.put(i, i * 10)

    print(f"Tamanho da tabela: {len(ht)}")

    # Teste de acesso
    print("\nTestando GET:")
    print("Chave 5:", ht.get(5))
    print("Chave 15:", ht.get(15))

    # Teste de update
    print("\nAtualizando chave 5...")
    ht.put(5, 999)
    print("Chave 5 atualizada:", ht.get(5))

    # Teste de delete
    print("\nRemovendo chave 10...")
    ht.delete(10)
    print("Chave 10:", ht.get(10))

    # Verificação após rehash
    print("\nVerificando integridade após rehash:")
    erro = False
    for i in range(20):
        if i != 10 and ht.get(i) is None:
            erro = True

    print("Tudo OK após rehash!" if not erro else "Erro encontrado!")

    print(f"\nTotal de comparações: {ht.comparacoes}")


if __name__ == "__main__":
    testar()

Inserindo elementos...
Rehash acontecendo...
Rehash acontecendo...
Tamanho da tabela: 20

Testando GET:
Chave 5: 50
Chave 15: 150

Atualizando chave 5...
Chave 5 atualizada: 999

Removendo chave 10...
Chave 10: None

Verificando integridade após rehash:
Tudo OK após rehash!

Total de comparações: 24



Encadeamento (Chaining)
Cada posição da tabela guarda uma lista encadeada
Resolve colisões armazenando múltiplos elementos no mesmo índice


Fator de carga (load factor)
	​
n = número de elementos
m = tamanho da tabela


Rehash

Quando α > 0.75:
dobra a capacidade
reinsere todos os elementos


Complexidade:

Rehash: O(n)
Amortizado: O(1) por operação


Comparações nos buckets

Melhor caso: O(1)
Médio: O(1 + α)
Pior caso: O(n)

Quanto maior o fator de carga → mais colisões → mais comparações

In [ ]:
# Questão 5

import random
from collections import deque


# NÓ DA ÁRVORE

class Node:
    def __init__(self, valor):
        self.valor = valor
        self.esq = None
        self.dir = None


# BST

class BST:
    def __init__(self):
        self.root = None

    def inserir(self, node, valor):
        if node is None:
            return Node(valor)

        if valor < node.valor:
            node.esq = self.inserir(node.esq, valor)
        else:
            node.dir = self.inserir(node.dir, valor)

        return node

    def inserir_lista(self, arr):
        for v in arr:
            self.root = self.inserir(self.root, v)


# BFS (NÍVEL) - FILA

def bfs(root):
    if not root:
        return []

    fila = deque([root])
    resultado = []
    operacoes = 0

    while fila:
        atual = fila.popleft()
        resultado.append(atual.valor)
        operacoes += 1

        if atual.esq:
            fila.append(atual.esq)
        if atual.dir:
            fila.append(atual.dir)

    return resultado, operacoes


# DFS (PROFUNDIDADE) - PILHA

def dfs(root):
    if not root:
        return []

    pilha = [root]
    resultado = []
    operacoes = 0

    while pilha:
        atual = pilha.pop()
        resultado.append(atual.valor)
        operacoes += 1

        # direita primeiro para visitar esquerda antes
        if atual.dir:
            pilha.append(atual.dir)
        if atual.esq:
            pilha.append(atual.esq)

    return resultado, operacoes


# TESTES

def testar():
    n = 15  # pequeno para visualizar
    vetor = [random.randint(1, 100) for _ in range(n)]

    print("Vetor gerado:")
    print(vetor)

    bst = BST()
    bst.inserir_lista(vetor)

    # BFS
    resultado_bfs, op_bfs = bfs(bst.root)
    print("\nBFS (nível):")
    print(resultado_bfs)
    print("Operações:", op_bfs)

    # DFS
    resultado_dfs, op_dfs = dfs(bst.root)
    print("\nDFS (profundidade):")
    print(resultado_dfs)
    print("Operações:", op_dfs)


if __name__ == "__main__":
    testar()

Vetor gerado:
[56, 92, 59, 37, 9, 76, 5, 35, 66, 65, 30, 82, 67, 81, 62]

BFS (nível):
[56, 37, 92, 9, 59, 5, 35, 76, 30, 66, 82, 65, 67, 81, 62]
Operações: 15

DFS (profundidade):
[56, 37, 9, 5, 35, 30, 92, 59, 76, 66, 65, 62, 67, 82, 81]
Operações: 15


BFS (Busca em Largura)

Usa fila (queue)
Percorre nível por nível
Complexidade: O(n)
Visita nós mais próximos da raiz primeiro


DFS (Busca em Profundidade)

Usa pilha (stack)
Vai até o fundo antes de voltar
Complexidade: O(n)
Explora caminhos completos


Análise de esforço

Ambos visitam todos os nós → O(n)
Diferença está na ordem de visita
BFS usa mais memória em árvores largas
DFS usa menos memória (em geral)


Possíveis usos

BFS

Encontrar menor caminho
Redes sociais (níveis de conexão)
Sistemas de rotas (GPS)

DFS

Resolver labirintos
Backtracking (ex: Sudoku)
Análise de grafos

In [ ]:
# Questão 6

# NÓ

class Node:
    def __init__(self, value):
        self.value = value
        self.next = None


# LISTA ENCADEADA

class SinglyLinkedList:
    def __init__(self):
        self.head = None
        self._size = 0

    # INSERT FIRST
    def insert_first(self, value):
        new = Node(value)
        new.next = self.head
        self.head = new
        self._size += 1

    # INSERT LAST
    def insert_last(self, value):
        new = Node(value)

        if not self.head:
            self.head = new
        else:
            current = self.head
            while current.next:
                current = current.next
            current.next = new

        self._size += 1

    # SEARCH
    def search(self, value):
        current = self.head
        pos = 0

        while current:
            if current.value == value:
                return pos
            current = current.next
            pos += 1

        return -1

    # DELETE (POR VALOR)
    def delete(self, value):
        current = self.head
        prev = None

        while current:
            if current.value == value:
                if prev:
                    prev.next = current.next
                else:
                    self.head = current.next
                self._size -= 1
                return True

            prev = current
            current = current.next

        print("Valor não encontrado.")
        return False

    # INSERT AT (POR POSIÇÃO)
    def insert_at(self, index, value):
        if index < 0 or index > self._size:
            print("Erro: índice inválido.")
            return

        if index == 0:
            self.insert_first(value)
            return

        new = Node(value)
        current = self.head

        for _ in range(index - 1):
            current = current.next

        new.next = current.next
        current.next = new
        self._size += 1

    # DELETE AT (POR POSIÇÃO)
    def delete_at(self, index):
        if index < 0 or index >= self._size:
            print("Erro: índice inválido.")
            return

        if index == 0:
            self.head = self.head.next
            self._size -= 1
            return

        current = self.head

        for _ in range(index - 1):
            current = current.next

        current.next = current.next.next
        self._size -= 1

    # LEN
    def __len__(self):
        return self._size

    # STR
    def __str__(self):
        values = []
        current = self.head

        while current:
            values.append(str(current.value))
            current = current.next

        return " -> ".join(values) if values else "Lista vazia"


# TESTES

def testar():
    lista = SinglyLinkedList()

    print("Inserindo elementos...")
    lista.insert_first(10)
    lista.insert_first(5)
    lista.insert_last(20)
    lista.insert_last(30)

    print("Lista:", lista)
    print("Tamanho:", len(lista))

    print("\nBuscando 20:", lista.search(20))
    print("Buscando 99:", lista.search(99))

    print("\nInserindo na posição 2...")
    lista.insert_at(2, 15)
    print("Lista:", lista)

    print("\nRemovendo valor 10...")
    lista.delete(10)
    print("Lista:", lista)

    print("\nRemovendo posição 1...")
    lista.delete_at(1)
    print("Lista:", lista)

    print("\nTeste de erro:")
    lista.insert_at(100, 50)
    lista.delete_at(100)

    print("\nTamanho final:", len(lista))


if __name__ == "__main__":
    testar()

Inserindo elementos...
Lista: 5 -> 10 -> 20 -> 30
Tamanho: 4

Buscando 20: 2
Buscando 99: -1

Inserindo na posição 2...
Lista: 5 -> 10 -> 15 -> 20 -> 30

Removendo valor 10...
Lista: 5 -> 15 -> 20 -> 30

Removendo posição 1...
Lista: 5 -> 20 -> 30

Teste de erro:
Erro: índice inválido.
Erro: índice inválido.

Tamanho final: 3


Análise Big O

| Operação       | Complexidade |
| -------------- | ------------ |
| insert_first   | O(1)         |
| insert_last    | O(n)         |
| search         | O(n)         |
| delete (valor) | O(n)         |
| insert_at      | O(n)         |
| delete_at      | O(n)         |
| **len**        | O(1)         |


A lista encadeada não possui acesso direto por índice
Operações no início são rápidas (O(1))
Operações que percorrem a lista são O(n)
Métodos com índice exigem caminhada até a posição


In [ ]:
# Questão 7

# NÓ

class Node:
    def __init__(self, value):
        self.value = value
        self.prev = None
        self.next = None


# DOUBLY LINKED LIST

class DoublyLinkedList:
    def __init__(self):
        self.head = None
        self.tail = None
        self._size = 0

    def is_empty(self):
        return self._size == 0

    def insert_first(self, value):
        new = Node(value)

        if self.is_empty():
            self.head = self.tail = new
        else:
            new.next = self.head
            self.head.prev = new
            self.head = new

        self._size += 1

    def insert_last(self, value):
        new = Node(value)

        if self.is_empty():
            self.head = self.tail = new
        else:
            new.prev = self.tail
            self.tail.next = new
            self.tail = new

        self._size += 1

    def delete_first(self):
        if self.is_empty():
            print("Erro: lista vazia")
            return None

        valor = self.head.value

        if self.head == self.tail:
            self.head = self.tail = None
        else:
            self.head = self.head.next
            self.head.prev = None

        self._size -= 1
        return valor

    def delete_last(self):
        if self.is_empty():
            print("Erro: lista vazia")
            return None

        valor = self.tail.value

        if self.head == self.tail:
            self.head = self.tail = None
        else:
            self.tail = self.tail.prev
            self.tail.next = None

        self._size -= 1
        return valor

    def __len__(self):
        return self._size

    def __str__(self):
        valores = []
        current = self.head
        while current:
            valores.append(str(current.value))
            current = current.next
        return " <-> ".join(valores) if valores else "Lista vazia"


# DEQUE

class Deque:
    def __init__(self):
        self.lista = DoublyLinkedList()

    def insert_left(self, value):
        self.lista.insert_first(value)

    def insert_right(self, value):
        self.lista.insert_last(value)

    def remove_left(self):
        return self.lista.delete_first()

    def remove_right(self):
        return self.lista.delete_last()

    def peek_left(self):
        return None if self.lista.is_empty() else self.lista.head.value

    def peek_right(self):
        return None if self.lista.is_empty() else self.lista.tail.value

    def __len__(self):
        return len(self.lista)

    def __str__(self):
        return str(self.lista)


# VERIFICAÇÃO DE INVARIANTES

def verificar_invariantes(lista):
    # verifica consistência dos ponteiros
    current = lista.head
    prev = None
    count = 0

    while current:
        if current.prev != prev:
            return False
        prev = current
        current = current.next
        count += 1

    # verifica tamanho
    return count == lista._size


# TESTES

def testar():
    dq = Deque()

    print("Inserindo nos dois lados...")
    dq.insert_left(10)
    dq.insert_right(20)
    dq.insert_left(5)
    dq.insert_right(30)

    print("Deque:", dq)
    print("Peek left:", dq.peek_left())
    print("Peek right:", dq.peek_right())

    print("\nRemovendo dos dois lados...")
    print("Remove left:", dq.remove_left())
    print("Remove right:", dq.remove_right())

    print("Deque:", dq)

    # Teste intensivo
    print("\nTeste de sequência alternada...")
    for i in range(10):
        if i % 2 == 0:
            dq.insert_left(i)
        else:
            dq.insert_right(i)

    print("Deque:", dq)

    for _ in range(5):
        dq.remove_left()
        dq.remove_right()

    print("Deque final:", dq)

    # Verificação
    valido = verificar_invariantes(dq.lista)
    print("\nInvariantes OK!" if valido else "Erro estrutural!")

    print("Tamanho final:", len(dq))


if __name__ == "__main__":
    testar()

Inserindo nos dois lados...
Deque: 5 <-> 10 <-> 20 <-> 30
Peek left: 5
Peek right: 30

Removendo dos dois lados...
Remove left: 5
Remove right: 30
Deque: 10 <-> 20

Teste de sequência alternada...
Deque: 8 <-> 6 <-> 4 <-> 2 <-> 0 <-> 10 <-> 20 <-> 1 <-> 3 <-> 5 <-> 7 <-> 9
Deque final: 10 <-> 20

Invariantes OK!
Tamanho final: 2


Invariantes verificados

Ponteiros prev e next consistentes
head.prev == None
tail.next == None
Contagem correta de nós

| Operação        | Complexidade |
| --------------- | ------------ |
| insert_left     | O(1)         |
| insert_right    | O(1)         |
| remove_left     | O(1)         |
| remove_right    | O(1)         |
| peek_left/right | O(1)         |
| is_empty        | O(1)         |


Lista duplamente encadeada permite acesso eficiente nas duas pontas
Deque aproveita isso para operações rápidas dos dois lados
Todas as operações principais são O(1)
Invariantes garantem integridade estrutural


In [ ]:
# Questão 8

# HASH TABLE (SIMPLIFICADA DA Q4)

class Node:
    def __init__(self, key, value):
        self.key = key
        self.value = value
        self.next = None


class HashTableChained:
    def __init__(self, capacidade=101):
        self.capacidade = capacidade
        self.tabela = [None] * capacidade

    def _hash(self, key):
        return hash(key) % self.capacidade

    def put(self, key, value):
        idx = self._hash(key)
        node = self.tabela[idx]

        while node:
            if node.key == key:
                node.value = value
                return
            node = node.next

        novo = Node(key, value)
        novo.next = self.tabela[idx]
        self.tabela[idx] = novo

    def get(self, key):
        idx = self._hash(key)
        node = self.tabela[idx]

        while node:
            if node.key == key:
                return node.value
            node = node.next

        return None


# DADOS (16 ITENS)

valores = [10, 40, 30, 50, 35, 40, 30, 45, 60, 55, 25, 20, 15, 10, 5, 50]
pesos   = [5,  4,  6,  3,  7,  2,  5,  6,  4,  3,  2,  1,  3,  2,  1,  8]
n = len(valores)
W = 20


# a) RECURSIVO PURO

chamadas = 0
subproblemas = set()

def knapsack_rec(i, capacidade):
    global chamadas
    chamadas += 1

    subproblemas.add((i, capacidade))

    if i == 0 or capacidade == 0:
        return 0

    if pesos[i-1] > capacidade:
        return knapsack_rec(i-1, capacidade)

    incluir = valores[i-1] + knapsack_rec(i-1, capacidade - pesos[i-1])
    excluir = knapsack_rec(i-1, capacidade)

    return max(incluir, excluir)


# b) MEMOIZATION (HASH)

chamadas_memo = 0
memo = HashTableChained()

def knapsack_memo(i, capacidade):
    global chamadas_memo
    chamadas_memo += 1

    chave = (i, capacidade)  # estado mínimo

    cached = memo.get(chave)
    if cached is not None:
        return cached

    if i == 0 or capacidade == 0:
        memo.put(chave, 0)
        return 0

    if pesos[i-1] > capacidade:
        resultado = knapsack_memo(i-1, capacidade)
    else:
        incluir = valores[i-1] + knapsack_memo(i-1, capacidade - pesos[i-1])
        excluir = knapsack_memo(i-1, capacidade)
        resultado = max(incluir, excluir)

    memo.put(chave, resultado)
    return resultado


# TESTE

def testar():
    global chamadas, chamadas_memo, subproblemas, memo

    print("=== RECURSIVO PURO ===")
    chamadas = 0
    subproblemas = set()

    resultado1 = knapsack_rec(n, W)

    print("Valor máximo:", resultado1)
    print("Chamadas recursivas:", chamadas)
    print("Subproblemas distintos:", len(subproblemas))


    print("\n=== COM MEMOIZATION ===")
    chamadas_memo = 0
    memo = HashTableChained()

    resultado2 = knapsack_memo(n, W)

    print("Valor máximo:", resultado2)
    print("Chamadas recursivas:", chamadas_memo)


if __name__ == "__main__":
    testar()

=== RECURSIVO PURO ===
Valor máximo: 295
Chamadas recursivas: 22679
Subproblemas distintos: 273

=== COM MEMOIZATION ===
Valor máximo: 295
Chamadas recursivas: 454


Análise de Complexidade

Recursivo puro
Complexidade: O(2ⁿ)
Muito lento
Recalcula vários subproblemas

Com memoization
Complexidade: O(n × W)
Muito mais eficiente
Cada subproblema resolvido uma única vez

| Método      | Complexidade | Chamadas    |
| ----------- | ------------ | ----------- |
| Recursivo   | O(2ⁿ)        | Muito alto  |
| Memoization | O(n·W)       | Muito menor |
